# Address to Coordinate Mapping of DVF Paris 2024–2025 Dataset via Base Adresse Nationale (BAN) API

**Source données:** dvf_paris_2024_2025.csv  
**API:** Géoplateforme Géocodage (IGN / BAN)  
**Endpoint (Address → Coordinates):** https://data.geopf.fr/geocodage/search  


## 1. Imports and Config

In [ ]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm
import json

# Configuration
INPUT = "../data/dvf_paris_2024_2025.csv"
OUTPUT = "dvf_paris_2024_2025_geocoded.csv"

BASE_URL  = "https://data.geopf.fr/geocodage/search"

# Score threshold below which a match is flagged for review
LOW_SCORE_THRESHOLD = 0.5

# Delay between requests in seconds (0.05 = 20 req/s, well within 50 req/s limit)
REQUEST_DELAY = 0.05

## 2. Load Data

In [ ]:
df = pd.read_csv(INPUT)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)

Shape: (120194, 20)
Columns: ['transaction_number', 'transaction_date', 'transaction_type', 'property_value', 'street_number', 'street_type', 'street_code', 'street_name', 'postal_code', 'commune', 'department_code', 'commune_code', 'section', 'plot_number', 'lot_count', 'property_type_code', 'property_type', 'surface_area', 'room_count', 'year']


,transaction_number,transaction_date,transaction_type,property_value,street_number,street_type,street_code,street_name,postal_code,commune,department_code,commune_code,section,plot_number,lot_count,property_type_code,property_type,surface_area,room_count,year
0,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020.0,PARIS 20,75.0,120.0,BM,133.0,2.0,2.0,Apartment,86.0,4.0,2024.0
1,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020.0,PARIS 20,75.0,120.0,BM,133.0,2.0,3.0,Outbuilding,0.0,0.0,2024.0
2,1,2024-01-04,Sale,1042000.0,16.0,RUE,2786,DE LA DHUIS,75020.0,PARIS 20,75.0,120.0,BM,133.0,1.0,3.0,Outbuilding,0.0,0.0,2024.0


In [ ]:
# Inspect address-relevant columns
address_columns = ["street_number", "street_type", "street_name", "postal_code", "commune"]
print("Missing values per address column:")
print(df[address_columns].isna().sum())
print()
df[address_columns].head(5)

Missing values per address column:
street_number    1
street_type      1
street_name      1
postal_code      1
commune          1
dtype: int64



,street_number,street_type,street_name,postal_code,commune
0,4.0,VLA,PERREUR,75020.0,PARIS 20
1,4.0,VLA,PERREUR,75020.0,PARIS 20
2,16.0,RUE,DE LA DHUIS,75020.0,PARIS 20
3,16.0,RUE,DE LA DHUIS,75020.0,PARIS 20
4,4.0,VLA,PERREUR,75020.0,PARIS 20


## 3. Build Address Query String

The BAN API takes a free-form address string via the "q" parameter, which needs to be created from the columns in the "dvf_paris_2024_2025" dataset.  

"street_number" - "street_type" - "street_name" -"postal_code"

"street_number" is stored as float (e.g. "4.0") and needs to be converted to string before concatenation.

In [ ]:
df = pd.read_csv(INPUT)

# street_number is stored as float (e.g. 4.0) and needs to be converted to string ("4") before concatenation
df["_street_num_str"] = df["street_number"].apply(
    lambda x: str(int(x)) if pd.notna(x) else ""
)
df["address"] = (
    df["_street_num_str"].str.strip() + " " +
    df["street_type"].str.strip()     + " " +
    df["street_name"].str.strip()     + " " +
    df["postal_code"].astype(str).str.strip()
).str.strip()

# Preview
df[["street_number", "street_type", "street_name", "postal_code", "address"]].head(5)

,street_number,street_type,street_name,postal_code,address
0,4.0,VLA,PERREUR,75020.0,4 VLA PERREUR 75020.0
1,4.0,VLA,PERREUR,75020.0,4 VLA PERREUR 75020.0
2,16.0,RUE,DE LA DHUIS,75020.0,16 RUE DE LA DHUIS 75020.0
3,16.0,RUE,DE LA DHUIS,75020.0,16 RUE DE LA DHUIS 75020.0
4,4.0,VLA,PERREUR,75020.0,4 VLA PERREUR 75020.0


## 4. Unique Addresses Only

The dataset contains many duplicate addresses (same building, multiple transactions).
Geocoding performed on unique addresses only to reduce API calls and runtime.



In [ ]:
unique_addresses = df["address"].drop_duplicates().reset_index(drop=True)

print(f"Total rows         : {len(df):,}")
print(f"Unique addresses   : {len(unique_addresses):,}")

Total rows         : 147,979
Unique addresses   : 30,696


## 5. API Response Check

In [ ]:
# Test with the first unique address before running all addresses
test_query = unique_addresses.iloc[0]
print("Test query:", test_query)

response = requests.get(BASE_URL, params={"q": test_query, "limit": 1})
print("Status code:", response.status_code)

data = response.json()
print("keys:", list(data.keys()))


Test query: 4 VLA PERREUR 75020.0
Status code: 200
keys: ['type', 'features', 'query']


In [ ]:
# Print the full raw JSON response to see the GeoJSON structure
response = requests.get(BASE_URL, params={"q": unique_addresses.iloc[0], "limit": 1})
data = response.json()

print(json.dumps(data, indent=2, ensure_ascii=False))

coords = data["features"][0]["geometry"]["coordinates"]
print(f"The API returns coordinates as [longitude, latitude]: lon={coords[0]}, lat={coords[1]}")

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.404984,
          48.868239
        ]
      },
      "properties": {
        "label": "4 Villa Perreur 75020 Paris",
        "score": 0.6649303557312252,
        "housenumber": "4",
        "id": "75120_7288_00004",
        "name": "4 Villa Perreur",
        "postcode": "75020",
        "citycode": "75120",
        "x": 656351.61,
        "y": 6863298.95,
        "city": "Paris",
        "district": "Paris 20e Arrondissement",
        "context": "75, Paris, Île-de-France",
        "type": "housenumber",
        "importance": 0.66206,
        "street": "Villa Perreur",
        "_type": "address"
      }
    }
  ],
  "query": "4 VLA PERREUR 75020.0"
}
The API returns coordinates as [longitude, latitude]: lon=2.404984, lat=48.868239


In [ ]:
# Inspect the response structure of the test request
feature = data["features"][0]
print("Geometry coordinates:", feature["geometry"]["coordinates"])
print()
print("Properties:")
for k, v in feature["properties"].items():
    print(f"  {k}: {v}")

Geometry coordinates: [2.404984, 48.868239]

Properties:
  label: 4 Villa Perreur 75020 Paris
  score: 0.6649303557312252
  housenumber: 4
  id: 75120_7288_00004
  name: 4 Villa Perreur
  postcode: 75020
  citycode: 75120
  x: 656351.61
  y: 6863298.95
  city: Paris
  district: Paris 20e Arrondissement
  context: 75, Paris, Île-de-France
  type: housenumber
  importance: 0.66206
  street: Villa Perreur
  _type: address


## 6. Address-to-Coordinate Mapping of Unique Addresses

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
coordinate_matched_addresses = []

for query in tqdm(unique_addresses, desc="Address-to-Coordinate Mapping"):
    params = {"q": query, "limit": 1}

    try:
        resp = requests.get(BASE_URL, params=params, timeout=10)
        resp.raise_for_status()
        features = resp.json().get("features", [])
    except requests.RequestException:
        # API request failed: store null values for this address to flag for manual review later
        coordinate_matched_addresses.append({
            "address"  : query,
            "lon"            : None,
            "lat"            : None,
            "matched_address": None,
            "match_score"    : None
        })
        continue

    if features:
        props  = features[0]["properties"]
        coords = features[0]["geometry"]["coordinates"]
        coordinate_matched_addresses.append({
            "address"  : query,
            "lon"            : coords[0],
            "lat"            : coords[1],
            "matched_address": props.get("label"),
            "match_score"    : props.get("score"),
        })
    else:
        coordinate_matched_addresses.append({
            "address"  : query,
            "lon"            : None,
            "lat"            : None,
            "matched_address": None,
            "match_score"    : None
        })

    time.sleep(REQUEST_DELAY)

print(f"\nTotal unique addresses matched to coordinates: {len(coordinate_matched_addresses):,}")

# Convert results to DataFrame
df_matched_addresses = pd.DataFrame(coordinate_matched_addresses)

# Save API outputs to Google Drive so they persist if the runtime ends
output_path = "../data/coordinate_matched_addresses.csv"
df_matched_addresses.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")


Address-to-Coordinate Mapping:   0%|          | 0/30696 [00:00<?, ?it/s]


Total unique addresses matched to coordinates: 30,696
Geocoding results saved to: /content/drive/MyDrive/Real Estate in Paris/Data/coordinate_matched_addresses.csv


In [ ]:
df_matched_addresses = pd.read_csv("../data/coordinate_matched_addresses.csv")

In [ ]:
df_matched_addresses.head(10)

,address,lon,lat,matched_address,match_score
0,4 VLA PERREUR 75020.0,2.404984,48.868239,4 Villa Perreur 75020 Paris,0.664930
1,16 RUE DE LA DHUIS 75020.0,2.404925,48.867963,16 Rue de la Dhuis 75020 Paris,0.819088
2,17 RUE DU ROI D ALGER 75018.0,2.347524,48.895295,17 Rue du Roi d'Alger 75018 Paris,0.823733
3,166 AV PARMENTIER 75010.0,2.370152,48.871446,166 Avenue Parmentier 75010 Paris,0.653717
4,76 RUE VIEILLE DU TEMPLE 75003.0,2.360309,48.859624,76 Rue Vieille du Temple 75003 Paris,0.827165
5,1 QUAI DE LA SEINE 75019.0,2.369590,48.884405,1 Quai de la Seine 75019 Paris,0.823347
6,1 RUE DE L ADJUDANT REAU 75020.0,2.405080,48.867304,1 Rue de l'Adjudant Réau 75020 Paris,0.825048
7,3 PAS BASFOUR 75002.0,2.351275,48.866222,3 Passage Basfour 75002 Paris,0.606950
8,24 RUE ARTHUR ROZIER 75019.0,2.391121,48.878727,24 Rue Arthur Rozier 75019 Paris,0.825326
9,13 AV DE CLICHY 75017.0,2.326634,48.884850,13 Avenue de Clichy 75017 Paris,0.649417


In [ ]:
# missing values
df_matched_addresses.isna().sum()

,0
address,1
lon,401
lat,401
matched_address,401
match_score,401


# One time fix for merge: remove trailing `.0` from Postal Code in Address

In [ ]:
import pandas as pd


Mounted at /content/drive


In [ ]:
df_matched_addresses = pd.read_csv("../data/coordinate_matched_addresses.csv")

In [ ]:
# Strip trailing ".0" from postal codes in address column and export again
# postal_code is stored as float (75020.0): remove .0 so address matches DVF merge key
df_matched_addresses["address"] = df_matched_addresses["address"].str.replace(".0", "", regex=False)

# Save to Google Drive
output_path = "../data/coordinate_matched_addresses.csv"
df_matched_addresses.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /content/drive/MyDrive/Real Estate in Paris/Data/coordinate_matched_addresses.csv
